# Recursos da interface (E2, caracterização)

Conta qubits, portas de dois qubits e profundidade das peças da interface em instâncias pequenas (custo unitário, m de 3 a 14, k ∈ {1, 2}, k < κ):

- **preparação A_k** (Dicke de Bärtschi–Eidenbenz, uniforme sobre F_k);
- **oráculo de tabela-verdade** (marca as M interdições ótimas; usado na demonstração em hardware);
- **oráculo aritmético sobre (x, α)**: γ_a = [α_i = 0 ∧ α_j = 1 ∧ x_a = 0] por portas multicontroladas, soma ponderada das capacidades (`WeightedSumGate`), comparação com τ (`IntegerComparatorGate`), fase e descomputação. Validado em `tests/test_oracle.py` em todos os estados de base de uma instância minúscula.

Duas contagens: **lógica** (base `cx`, `u`, sem restrição de conectividade) e **transpilada** para o `FakeFez` (Heron, 156 qubits, nível de otimização 1).

Colunas analíticas: a versão (x, α) busca sobre |F_k| · 2^(n−2) pares, com O(√(|F_k| · 2^(n−2))) consultas; ela só supera a enumeração clássica de F_k (|F_k| avaliações) quando 2^(n−2) < |F_k|.

## 1. Imports, instâncias e constantes

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from qiskit_ibm_runtime.fake_provider import FakeFez

from qmfnip.grover import search_space
from qmfnip.instances import Scenario, full_cut_cost, max_flow
from qmfnip.oracle import arithmetic_oracle, circuit_resources, phase_oracle, state_preparation
from qmfnip.outputs import ExperimentOutput, capture_environment

In [ ]:
SCENARIO = Scenario(
    "random",
    sizes=(3, 4, 5, 6, 7),
    budgets=(1, 2),
    seeds=(0,),
    params={"arc_factor": 2.0, "cost_range": (1, 1)},
    skip_degenerate=True,
)
BACKEND = FakeFez()
SEED = 0
instances = list(SCENARIO.instances())
[(instance.instance_id, instance.m, instance.budget) for instance in instances]

## 2. Execução: montagem e contagem dos circuitos

In [ ]:
output = ExperimentOutput.create(["dicke", "truth_table", "arithmetic"], SCENARIO.family, [instance.n for instance in instances], 1)
output.write_json("environment.json", capture_environment())
output.update_manifest({"status": "running", "scenario": SCENARIO.to_dict(), "backend": BACKEND.name, "seed": SEED})
rows = []
for instance in instances:
    output.write_instance(instance)
    space = search_space(instance)
    threshold = space.optimum + 1
    marked = space.interdictions[space.optimal_indices].astype(int).tolist()
    circuits = {
        "dicke_preparation": state_preparation(instance.m, instance.budget),
        "truth_table_oracle": phase_oracle(instance.m, marked),
        "arithmetic_oracle": arithmetic_oracle(instance, threshold),
    }
    for name, circuit in circuits.items():
        for backend in (None, BACKEND):
            resources = circuit_resources(circuit, backend, seed=SEED)
            resources.pop("operations")
            rows.append(
                {
                    "instance_id": instance.instance_id, "n": instance.n, "m": instance.m, "k": instance.budget,
                    "kappa": full_cut_cost(instance), "N": space.size, "M": space.optimal_count,
                    "flow_without_interdiction": max_flow(instance), "optimal_flow": space.optimum,
                    "circuit": name, **resources,
                }
            )
resources = pd.DataFrame(rows)
output.write_table("resources", resources)
output.update_manifest({"status": "completed"})
resources.head()

## 3. Avaliação: tabela T3 e colunas analíticas

In [ ]:
t3 = resources.pivot_table(
    index=["instance_id", "n", "m", "k", "N", "M"],
    columns=["circuit", "backend"],
    values=["logical_qubits", "physical_qubits", "two_qubit_gates", "circuit_depth"],
    aggfunc="first",
)
t3 = t3.sort_index(level=["m", "k"])
output.write_table("t3_resources_wide", t3.reset_index())
t3[["logical_qubits", "two_qubit_gates"]]

In [ ]:
analytic = resources.drop_duplicates("instance_id")[["instance_id", "n", "m", "k", "N", "M"]].copy()
analytic["search_qubits_x"] = analytic["m"]
analytic["search_qubits_x_alpha"] = analytic["m"] + analytic["n"] - 2
analytic["pairs"] = analytic["N"] * 2.0 ** (analytic["n"] - 2)
analytic["queries_x"] = (math.pi / 4) * analytic["N"].pow(0.5)
analytic["queries_x_alpha"] = (math.pi / 4) * analytic["pairs"].pow(0.5)
analytic["classical_evaluations"] = analytic["N"]
analytic["x_alpha_beats_enumeration"] = 2.0 ** (analytic["n"] - 2) < analytic["N"]
arithmetic = resources[(resources["circuit"] == "arithmetic_oracle") & (resources["backend"] == "logical")].set_index("instance_id")
analytic["arithmetic_oracle_qubits"] = analytic["instance_id"].map(arithmetic["logical_qubits"])
analytic["arithmetic_oracle_two_qubit_gates"] = analytic["instance_id"].map(arithmetic["two_qubit_gates"])
analytic = analytic.sort_values(["m", "k"])
output.write_table("t3_analytic", analytic)
analytic

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis, backend in zip(axes, ("logical", BACKEND.name)):
    for index, (name, group) in enumerate(resources[resources["backend"] == backend].groupby("circuit", sort=False)):
        group = group.sort_values(["m", "k"])
        axis.plot(group["m"] + 0.08 * (group["k"] - 1), group["two_qubit_gates"], marker="o", ls="", color=f"C{index}", label=name)
    axis.set_yscale("log")
    axis.set_xlabel("arcs m")
    axis.set_title(f"two-qubit gates ({backend})")
    axis.grid(alpha=0.3, which="both", lw=0.4)
axes[0].set_ylabel("two-qubit gates")
axes[1].legend(fontsize=8)
figure.tight_layout()
figure.savefig(output.figure_path("resources_two_qubit_gates"), bbox_inches="tight")